In [1]:
!pip install -U langchain langchain_mistralai mistralai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 4.7 MB/s eta 0:00:00


In [2]:
import ast
import json
import requests
import pandas as pd
import numpy as np
import aiohttp, asyncio
import time
import nest_asyncio

from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio

import os
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field
from typing import List, Optional

from langchain_mistralai import ChatMistralAI
from mistralai import Mistral

MISTRAL_API = os.getenv('MISTRAL_API')
if MISTRAL_API:
    print('Found token')

In [3]:
from google.colab import userdata
MISTRAL_API = userdata.get('MISTRAL_API_KEY')
if MISTRAL_API:
    print('Found token')

Found token


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Data preparation

In [5]:
datapath = '/content/drive/MyDrive/ITMO/intro_to_llm/llm_project/data/'

In [23]:
months2num = {
    'January': str(1), 'February': str(2), 'March': str(3),
    'April': str(4), 'May': str(5), 'June': str(6),
    'July': str(7), 'August': str(8), 'September' : str(9),
    'October': str(10), 'November': str(11), 'December': str(12),
    'Jan': str(1), 'Feb': str(2), 'Mar': str(3),
    'Apr': str(4), 'May': str(5), 'Jun': str(6),
    'Jul': str(7), 'Aug': str(8), 'Sep' : str(9),
    'Oct': str(10), 'Nov': str(11), 'Dec': str(12)
}

In [70]:
def prepare_genres(genres):
    if genres is np.nan:
        return np.nan

    genres = list(ast.literal_eval(genres).values())
    return ','.join(genres)


headers = [
    'wikipedia_ID', 'freebase_ID',
    'title', 'author',
    'publication_date', 'genre',
    'description'
]
book_summaries = pd.read_csv(
    datapath+"booksummaries.txt", sep='\t', names=headers
)
book_summaries['genre'] = book_summaries['genre'].apply(prepare_genres)
book_summaries['page_count'] = np.nan

book_dataset = pd.read_csv(datapath+'BooksDatasetClean.csv')
book_dataset['Publish Date'] = book_dataset.apply(
    lambda row:
        months2num.get(row['Publish Date (Month)']) + ' ' + str(row['Publish Date (Year)'])
        if row['Publish Date (Month)'] is not np.nan and row['Publish Date (Year)'] is not np.nan else np.nan,

    axis=1
)
book_dataset.rename(
    columns={
        'Title': 'title',
        'Authors': 'author',
        'Description': 'description',
        'Category': 'genre',
        'Publish Date': 'publication_date'
    },
    inplace=True
)

# delete "By" in the beggining of author name
book_dataset['author'] = book_dataset['author'].apply(lambda s: s[3:])
book_dataset['page_count'] = np.nan

good_reads = pd.read_csv(datapath+'GoodReads_100k_books.csv')
good_reads.rename(
    columns={
        'desc': 'description',
        'pages': 'page_count'
    },
    inplace=True
)
good_reads['publication_date'] = np.nan

google_books = pd.read_csv(datapath+'google_books_1299.csv')
google_books.rename(
    columns={
        'generes': 'genre',
        'published_date': 'publication_date',
    },
    inplace=True
)
google_books['genre'] = google_books['genre'].apply(lambda genre: genre.replace(' &amp', ''))

def prepare_date_google(date):
    date = date.split()
    if len(date) < 2:
        return np.nan
    month, day, year = date
    month = months2num.get(month)
    return month + ' ' + day + ' ' + year
google_books['publication_date'] = google_books['publication_date'].apply(prepare_date_google)

Недостающие факты:
- описание
- дата

In [71]:
def prepare_date(date, format='%m %Y'):
    if date is np.nan:
        return date
    date = datetime.strptime(date, format)
    date = datetime.strftime(date, '%Y-%m-%d')
    return date

book_dataset['publication_date'] = book_dataset.publication_date.apply(prepare_date)
google_books['publication_date'] = google_books.publication_date.apply(lambda date: prepare_date(date, '%m %d, %Y'))
book_summaries['publication_date'] = book_summaries['publication_date'].apply(
    lambda date: prepare_date(date, '%Y')
    if date is not np.nan and '-' not in date
    else date
)


In [26]:
book_summaries.head()

,wikipedia_ID,freebase_ID,title,author,publication_date,genre,description,page_count
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"Roman à clef,Satire,Children's literature,Spec...","Old Major, the old boar on the Manor Farm, ca...",NaN
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962-01-01,"Science Fiction,Novella,Speculative fiction,Ut...","Alex, a teenager living in near-future Englan...",NaN
2,986,/m/0ldx,The Plague,Albert Camus,1947-01-01,"Existentialism,Fiction,Absurdist fiction,Novel",The text of The Plague is divided into five p...,NaN
3,1756,/m/0sww,An Enquiry Concerning Human Understanding,David Hume,NaN,NaN,The argument of the Enquiry proceeds by a ser...,NaN
4,2080,/m/0wkt,A Fire Upon the Deep,Vernor Vinge,NaN,"Hard science fiction,Science Fiction,Speculati...",The novel posits that space around the Milky ...,NaN


In [27]:
book_dataset.head()

,title,author,description,genre,Publisher,Price Starting With ($),Publish Date (Month),Publish Date (Year),publication_date,page_count
0,Goat Brothers,"Colton, Larry",NaN,"History , General",Doubleday,8.79,January,1993,1993-01-01,NaN
1,The Missing Person,"Grumbach, Doris",NaN,"Fiction , General",Putnam Pub Group,4.99,March,1981,1981-03-01,NaN
2,Don't Eat Your Heart Out Cookbook,"Piscatella, Joseph C.",NaN,"Cooking , Reference",Workman Pub Co,4.99,September,1983,1983-09-01,NaN
3,When Your Corporate Umbrella Begins to Leak: A...,"Davis, Paul D.",NaN,NaN,Natl Pr Books,4.99,April,1991,1991-04-01,NaN
4,Amy Spangler's Breastfeeding : A Parent's Guide,"Spangler, Amy",NaN,NaN,Amy Spangler,5.32,February,1997,1997-02-01,NaN


In [ ]:
good_reads.head()

,author,bookformat,description,genre,img,isbn,isbn13,link,page_count,rating,reviews,title,totalratings,publication_date
0,Laurence M. Hauptman,Hardcover,Reveals that several hundred thousand Indians ...,"History,Military History,Civil War,American Hi...",https://i.gr-assets.com/images/S/compressed.ph...,002914180X,9.78E+12,https://goodreads.com/book/show/1001053.Betwee...,0,3.52,5,Between Two Fires: American Indians in the Civ...,33,NaN
1,"Charlotte Fiell,Emmanuelle Dirix",Paperback,Fashion Sourcebook - 1920s is the first book i...,"Couture,Fashion,Historical,Art,Nonfiction",https://i.gr-assets.com/images/S/compressed.ph...,1906863482,9.78E+12,https://goodreads.com/book/show/10010552-fashi...,576,4.51,6,Fashion Sourcebook 1920s,41,NaN
2,Andy Anderson,Paperback,The seminal history and analysis of the Hungar...,"Politics,History",https://i.gr-assets.com/images/S/compressed.ph...,948984147,9.78E+12,https://goodreads.com/book/show/1001077.Hungar...,124,4.15,2,Hungary 56,26,NaN
3,Carlotta R. Anderson,Hardcover,"""All-American Anarchist"" chronicles the life a...","Labor,History",https://i.gr-assets.com/images/S/compressed.ph...,814327079,9.78E+12,https://goodreads.com/book/show/1001079.All_Am...,324,3.83,1,All-American Anarchist: Joseph A. Labadie and ...,6,NaN
4,Jean Leveille,NaN,"Aujourdâ€™hui, lâ€™oiseau nous invite Ã sa ta...",NaN,https://i.gr-assets.com/images/S/compressed.ph...,2761920813,NaN,https://goodreads.com/book/show/10010880-les-o...,177,4.00,1,Les oiseaux gourmands,1,NaN


In [ ]:
google_books.head()

,Unnamed: 0,title,author,rating,voters,price,currency,description,publisher,page_count,genre,ISBN,language,publication_date
0,0,Attack on Titan: Volume 13,Hajime Isayama,4.6,428,43.28,SAR,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,192,none,9781612626864,English,2014-07-31
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288,"Fiction , Mystery, Detective , Cozy , General",9780758272799,English,2007-07-01
2,2,The Art of Super Mario Odyssey,Nintendo,3.9,9,133.85,SAR,Take a globetrotting journey all over the worl...,Dark Horse Comics,368,"Games, Activities , Video, Electronic",9781506713816,English,2019-11-05
3,3,Getting Away Is Deadly: An Ellie Avery Mystery,Sara Rosett,4.0,10,26.15,SAR,"With swollen feet and swelling belly, pregnant...",Kensington Publishing Corp.,320,none,9781617734076,English,2009-03-01
4,4,"The Painted Man (The Demon Cycle, Book 1)",Peter V. Brett,4.5,577,28.54,SAR,The stunning debut fantasy novel from author P...,HarperCollins UK,544,"Fiction , Fantasy , Dark Fantasy",9780007287758,English,2009-01-08


In [ ]:
headers = [
    'title', 'author', 'publication_date',
    'description', 'page_count'
]

In [ ]:
all_dfs = [book_summaries[headers], book_dataset[headers], good_reads[headers], google_books[headers]]
books_dfs = pd.concat(all_dfs, ignore_index=True)
books_dfs.dropna(subset=['title'], inplace=True)

In [ ]:
print('Наны авторов:', books_dfs.author.isna().sum() / books_dfs.shape[0])
print('Наны названий:', books_dfs.title.isna().sum() / books_dfs.shape[0])
print('Наны описаний:', books_dfs.description.isna().sum() / books_dfs.shape[0])
# print('Наны жанров:', books_dfs.genre.isna().sum() / books_dfs.shape[0])
print('Наны дат:', books_dfs.publication_date.isna().sum() / books_dfs.shape[0])

Наны авторов: 0.01078218359587181
Наны названий: 0.0
Наны описаний: 0.17939978272677892
Наны дат: 0.47805087814593517


### Генерируем недостающее

In [ ]:
def get_model(api_key: str):
    llm = ChatMistralAI(
        model="mistral-small-latest",
        max_retries=2,
        api_key=api_key
    )
    llm.verbose = False
    return llm


def get_prompt(user_content):
    message = [
        {"role": "system", "content": "You are an AI assistant"},
        {"role": "user", "content": user_content}
    ]
    return message

llm = get_model(MISTRAL_API)

In [ ]:
class BookMetadata(BaseModel):
    title: str = Field(..., description="Title of the book")
    author: List[str] = Field(..., description="List of authors.")
    publication_date: str = Field(..., description="Publication date in YYYY-MM-DD format.")
    description: str = Field(..., description="Brief description of the book")
    page_count: Optional[int] = Field(..., description="Approximate number of pages.")

In [ ]:
def get_prompt(title, author, description, publication_date, page_count):
    author = author if author is not np.nan else 'No author'
    description = description if description is not np.nan else 'No description'
    publication_date = publication_date if publication_date is not np.nan else 'No publication date'
    page_count = page_count if page_count is not np.nan else 'No page count'

    prompt = f"""Find the missing information:
title: {title}
author: {author}
description: {description}
publication_date: {publication_date}
page_count: {page_count}
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a specialist in books. Your work is to find relevant information about a book. "
                "You are given a known information about a book a should return this information and write the metadata which is absent.\n"
                "Return answer by a given scheme, no additional text. Example of valid answer:\n"
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
    return messages

In [ ]:
# title, author, publication_date, description, pages_count = books_dfs.values[0]
# messages = get_prompt(title, author, description, publication_date, pages_count)
# client = Mistral(api_key=MISTRAL_API)
# completion = client.chat.parse(
#     model="mistral-small-latest",
#     messages=messages,
#     response_format=BookMetadata,
#     max_tokens=2000,
#     temperature=0.3
# )

# book_info = completion.choices[0].message.parsed
# json.loads(book_info.model_dump_json())

### Google API

In [ ]:
client = Mistral(api_key=MISTRAL_API)

base_url = "https://www.googleapis.com/books/v1/volumes"

def get_book_by_title_author(title: str, author: str = None):
    """
    Поиск книги по названию и автору
    """
    query = f'intitle:"{title}"'
    if author:
        query += f' inauthor:"{author}"'

    result = search_books(query, max_results=1)

    if result.get('items'):
        return result['items']
    return None

def search_books(query: str, max_results: int = 10):
    """
    Поиск книг по запросу
    """
    params = {
        'q': query,
        'maxResults': max_results,
        'printType': 'books'
    }

    response = requests.get(base_url, params=params)
    response.raise_for_status()

    return response.json()

def get_answers(row_values, num_retries=3):
    # print(row_values)
    title, author, publication_date, description, pages_count = row_values
    messages = get_prompt(title, author, description, publication_date, pages_count)

    for num_retry in range(num_retries):
        try:
            completion = client.chat.parse(
                model="mistral-small-latest",
                messages=messages,
                response_format=BookMetadata,
                max_tokens=2000,
                temperature=0.3
            )
        except Exception as e:
            if num_retry == num_retries-1:
                print('Retries are out, return unchanged values')
                col_names = ['title', 'author', 'publication_date', 'description', 'page_count']
                col_values = row_values
                return col_names, col_values
        else:
            break

    book_info = completion.choices[0].message.parsed
    book_info = json.loads(book_info.model_dump_json())
    book_info['author'] = ','.join(book_info['author'])
    col_names = list(book_info.keys())
    col_values = list(book_info.values())
    return col_names, col_values

def fill_nan(df, num_retries=3):
    llm_calls = 0
    for num_row, row in tqdm(df.iterrows(), total=len(df)):
        title = row.title
        for num_retry in range(num_retries):
            try:
                results = get_book_by_title_author(title=title)
            except Exception as e:
                if num_retry == num_retries-1:
                    print('Row:', i)
                    print('Error:', e)
                    results = None
                    break
            else:
                break

        results_en = [res['volumeInfo'] for res in results if res['volumeInfo']['language']=='en'] if results else None
        if not results:
            # col_names, col_values = get_answers(row.values)
            # df.loc[num_row, col_names] = col_values
            # llm_calls += 1
            # continue
            result = {}

        elif not results_en:
            # col_names, col_values = get_answers(row.values)
            # df.loc[num_row, col_names] = col_values
            # llm_calls += 1
            # continue
            result = {}

        else:
            result = results_en[0]

        nan_cols = []
        nan_values = []
        if pd.isna(row.author):
            nan_cols.append('author')
            nan_values.append(','.join(result.get('authors', ['Not known'])))

        if pd.isna(row.description):
            nan_cols.append('description')
            nan_values.append(result.get('description', 'Not known'))

        if pd.isna(row.page_count):
            nan_cols.append('page_count')
            nan_values.append(result.get('pageCount', 'Not known'))

        if pd.isna(row.publication_date):
            nan_cols.append('publication_date')
            date = result.get('publishedDate', 'Not known')
            if date != 'Not known':
                if '-' not in date:
                    date = prepare_date(date, '%Y')
            nan_values.append(date)

        df.loc[num_row, nan_cols] = nan_values

            # if 'Not known' in nan_values:
            #     col_names, col_values = get_answers(df.loc[num_row].values)
            #     df.loc[num_row, col_names] = col_values
            #     llm_calls += 1

    print('llm_calls', llm_calls)

    return df

In [ ]:
df_sample = books_dfs.sample(100, random_state=10)

In [ ]:
print(books_dfs.columns)
print(df_sample.columns)

Index(['title', 'author', 'publication_date', 'description', 'page_count'], dtype='object')
Index(['title', 'author', 'publication_date', 'description', 'page_count'], dtype='object')


In [ ]:
print('Наны авторов:', df_sample.author.isna().sum() / df_sample.shape[0])
print('Наны названий:', df_sample.title.isna().sum() / df_sample.shape[0])
print('Наны описаний:', df_sample.description.isna().sum() / df_sample.shape[0])
print('Наны дат:', df_sample.publication_date.isna().sum() / df_sample.shape[0])

Наны авторов: 0.04
Наны названий: 0.0
Наны описаний: 0.17
Наны дат: 0.38


In [ ]:
books_dfs = books_dfs.drop_duplicates('title')

In [ ]:
# books_dfs = fill_nan(books_dfs)

In [ ]:
print('Наны авторов:', df_sample.author.isna().sum() / df_sample.shape[0])
print('Наны названий:', df_sample.title.isna().sum() / df_sample.shape[0])
print('Наны описаний:', df_sample.description.isna().sum() / df_sample.shape[0])
print('Наны дат:', df_sample.publication_date.isna().sum() / df_sample.shape[0])
print('Наны страниц:', df_sample.page_count.isna().sum() / df_sample.shape[0])

Наны авторов: 0.04
Наны названий: 0.0
Наны описаний: 0.17
Наны дат: 0.38
Наны страниц: 0.65


### Async

In [ ]:
time.sleep(1)

In [ ]:
client = Mistral(api_key=MISTRAL_API)

base_url = "https://www.googleapis.com/books/v1/volumes"

async def search_books_async(query: str, max_results: int = 10, session: aiohttp.ClientSession = None):
    """
    Асинхронный поиск книг по запросу
    """
    params = {
        'q': query,
        'maxResults': max_results,
        'printType': 'books'
    }

    if session is None:
        async with aiohttp.ClientSession() as temp_session:
            async with temp_session.get(base_url, params=params) as response:
                response.raise_for_status()
                return await response.json()
    else:
        async with session.get(base_url, params=params) as response:
            response.raise_for_status()
            return await response.json()

async def get_book_by_title_author_async(title: str, author: str = None, session: aiohttp.ClientSession = None):
    """
    Асинхронный поиск книги по названию и автору
    """
    query = f'intitle:"{title}"'
    if author:
        query += f' inauthor:"{author}"'

    result = await search_books_async(query, max_results=1, session=session)

    if result.get('items'):
        return result['items']
    return None

async def process_single_row(session: aiohttp.ClientSession, row: pd.Series, num_row: int, num_retries: int = 3):
    """
    Асинхронная обработка одной строки
    """
    title = row.title
    error = None

    # Асинхронный поиск книги в Google Books
    for num_retry in range(num_retries):
        try:
            results = await get_book_by_title_author_async(title=title, session=session)
            break
        except Exception as e:
            if num_retry < num_retries-1:
                time.sleep(0.5)
            elif num_retry == num_retries-1:
                # print(f'Row {num_row}: Error {e}')
                results = None
                error = num_row
            continue
    else:
        results = None

    # Фильтруем английские книги
    results_en = [res['volumeInfo'] for res in results if res['volumeInfo'].get('language') == 'en'] if results else None

    if not results:
        result = {}
    elif not results_en:
        result = {}
    else:
        result = results_en[0]

    # Определяем какие колонки нужно заполнить
    nan_cols = []
    nan_values = []

    if pd.isna(row.author):
        nan_cols.append('author')
        nan_values.append(','.join(result.get('authors', ['Not known'])))

    if pd.isna(row.description):
        nan_cols.append('description')
        nan_values.append(result.get('description', 'Not known'))

    if pd.isna(row.page_count):
        nan_cols.append('page_count')
        nan_values.append(result.get('pageCount', 'Not known'))

    if pd.isna(row.publication_date):
        nan_cols.append('publication_date')
        date = result.get('publishedDate', 'Not known')
        if date != 'Not known':
            if ('-' not in date) and ('?' not in date):
                try:
                    date = prepare_date(date, '%Y')
                except Exception as e:
                    print('Date error:', e)
                    pass
        nan_values.append(date)

    return num_row, nan_cols, nan_values, error

async def fill_nan_async(df: pd.DataFrame, num_retries: int = 3, max_concurrent: int = 10) -> pd.DataFrame:
    """
    Асинхронное заполнение NaN значений
    """
    llm_calls = 0
    semaphore = asyncio.Semaphore(max_concurrent)  # Ограничиваем concurrent запросы

    async def process_row_with_limit(session, row, num_row):
        async with semaphore:
            return await process_single_row(session, row, num_row, num_retries)

    async with aiohttp.ClientSession() as session:
        # Создаем задачи для всех строк
        tasks = []
        for num_row, row in df.iterrows():
            task = process_row_with_limit(session, row, num_row)
            tasks.append(task)

        # Выполняем все задачи с прогресс-баром
        results = await tqdm_asyncio.gather(
            *tasks,
            desc="🔄 Processing rows",
            total=len(df)
        )

        # Применяем результаты к DataFrame
        all_error_ids = []
        for num_row, nan_cols, nan_values, error in results:
            if nan_cols and nan_values:
                df.loc[num_row, nan_cols] = nan_values
            if error:
                all_error_ids.append(error)

    print('llm_calls', llm_calls)
    return df, all_error_ids

In [ ]:
def fill_nan(df: pd.DataFrame, num_retries: int = 3, max_concurrent: int = 10) -> pd.DataFrame:
    """
    Обертка с nest_asyncio для Jupyter
    """
    # Применяем nest_asyncio только если нужно
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            nest_asyncio.apply()
            return loop.run_until_complete(fill_nan_async(df, num_retries, max_concurrent))
    except RuntimeError:
        pass

    # Для обычных скриптов
    return asyncio.run(fill_nan_async(df, num_retries, max_concurrent))

In [ ]:
books_dfs_nans = books_dfs.loc[books_dfs.description.isna()].copy()
books_dfs_nans = books_dfs_nans.reset_index().drop(columns=['index'])

books_dfs_no_nans = books_dfs.loc[~books_dfs.description.isna()].copy()

In [ ]:
books_dfs_nans.shape

(37718, 5)

In [ ]:
books_dfs_nans

,title,author,publication_date,description,page_count
0,Goat Brothers,"Colton, Larry",1993-01-01,NaN,NaN
1,The Missing Person,"Grumbach, Doris",1981-03-01,NaN,NaN
2,Don't Eat Your Heart Out Cookbook,"Piscatella, Joseph C.",1983-09-01,NaN,NaN
3,When Your Corporate Umbrella Begins to Leak: A...,"Davis, Paul D.",1991-04-01,NaN,NaN
4,Amy Spangler's Breastfeeding : A Parent's Guide,"Spangler, Amy",1997-02-01,NaN,NaN
...,...,...,...,...,...
37713,the priory and parish church of st mary Chepstow,st mary Chepstow,NaN,NaN,14.0
37714,Say say say(#2),Mi-Ri Hwang,NaN,NaN,0.0
37715,(FREE SAMPLE) CTET English & Hindi Language 9 ...,Disha Experts,2019-11-06,NaN,23.0
37716,1 Sample Paper for CBSE Class 10 Science 2020 ...,Disha Experts,2019-10-21,NaN,25.0


In [ ]:
df_filled, all_error_ids = fill_nan(books_dfs_nans, num_retries=3, max_concurrent=3)

🔄 Processing rows:   4%|▎         | 1335/37718 [10:38<4:11:03,  2.42it/s]

Date error: unconverted data remains: *


🔄 Processing rows:   7%|▋         | 2812/37718 [23:10<4:39:45,  2.08it/s]

Date error: unconverted data remains: *


🔄 Processing rows:  27%|██▋       | 10228/37718 [1:23:09<2:27:06,  3.11it/s]

...

Date error: unconverted data remains: *


🔄 Processing rows:  95%|█████████▍| 35673/37718 [4:47:24<15:47,  2.16it/s]

Date error: unconverted data remains: *


🔄 Processing rows: 100%|██████████| 37718/37718 [5:03:32<00:00,  2.07it/s]
/tmp/ipython-input-2027139271.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Not known' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[num_row, nan_cols] = nan_values


llm_calls 0


In [ ]:
df_filled.to_excel(datapath + 'added_description.xlsx')

In [ ]:
df_filled_errors = df_filled.loc[all_error_ids]
df_filled_errors, all_error_ids = fill_nan(df_filled_errors, num_retries=3, max_concurrent=3)

🔄 Processing rows: 100%|██████████| 8031/8031 [1:06:24<00:00,  2.02it/s]

llm_calls 0


In [ ]:
has_description = pd.concat([books_dfs_no_nans, df_filled], ignore_index=True)

In [ ]:
has_description.to_excel(datapath + 'book_data_prepared.xlsx')

### Clean descriptions

In [77]:
books_prepared = pd.read_excel(datapath + 'book_data_prepared.xlsx', index_col=0)

In [78]:
books_prepared.head(1)

,title,author,publication_date,description,page_count
0,Animal Farm,George Orwell,1945-08-17,"Old Major, the old boar on the Manor Farm, ca...",NaN


In [79]:
books_prepared = books_prepared.loc[books_prepared.description!='Not known'].reset_index().drop(columns=['index'])

In [17]:
books_prepared

,title,author,publication_date,description,page_count
0,Animal Farm,George Orwell,1945-08-17,"Old Major, the old boar on the Manor Farm, ca...",NaN
1,A Clockwork Orange,Anthony Burgess,1962-01-01,"Alex, a teenager living in near-future Englan...",NaN
2,The Plague,Albert Camus,1947-01-01,The text of The Plague is divided into five p...,NaN
3,An Enquiry Concerning Human Understanding,David Hume,NaN,The argument of the Enquiry proceeds by a ser...,NaN
4,A Fire Upon the Deep,Vernor Vinge,NaN,The novel posits that space around the Milky ...,NaN
...,...,...,...,...,...
175545,Coaching Outside the Box,Paul Mairs,2012-01-01,This book demonstrates what coaches should and...,171
175546,The women's wheel of life,"Elizabeth Davis,Carol Leonard",1997-04,For women who have found identity in nonconven...,240
175547,"Gilded New York: Design, Fashion, and Society","Jeannine Falino,Phyllis Magidson,Nina Gray,Don...",2013-11-05,The Gilded Years of the late nineteenth centur...,240
175548,Student Power in World Evangelism,David M. Howard Jr.,1970-01-01,"When students decide to act, things happen. Th...",129


In [74]:
def comma_split(name):
    surname, name = name.split(',')
    return name + ' ' + surname

def author_preparation(authors):
    if authors.count(',')==0:
        return authors

    if authors.count(',')==1:
        return comma_split(authors)

    if 'and' in authors:
        authors = authors.split(' and ')
        if len(authors) == 2:
            if authors[0].count(',')==1:
                name1 = comma_split(authors[0])
            else:
                name1 = None

            if authors[1].count(',')==1:
                name2 = comma_split(authors[1])
            else:
                name2 = None

            if name1 and name2:
                return name1 + ',' + name2

        else:
            authors = ','.join(authors)

    return 'no'

book_dataset.author = book_dataset.author.apply(author_preparation)

In [75]:
book_dataset.author.sample(10)

,author
96172,Elm Hill
98848,Scott Brown
37720,Cathy East Dubowski
53467,Garry Hogg
58812,Lito Tejada-Flores
99419,Disney Enterprise (COR)
26287,Philip Langdon
31161,no
11903,Jeff Shapiro
41396,Steve Lopez


In [94]:
book_dataset = book_dataset.loc[(book_dataset.author != 'no')]

In [95]:
books_prepared

,title,author,publication_date,description,page_count
0,Animal Farm,George Orwell,1945-08-17,"Old Major, the old boar on the Manor Farm, ca...",NaN
1,A Clockwork Orange,Anthony Burgess,1962-01-01,"Alex, a teenager living in near-future Englan...",NaN
2,The Plague,Albert Camus,1947-01-01,The text of The Plague is divided into five p...,NaN
3,An Enquiry Concerning Human Understanding,David Hume,NaN,The argument of the Enquiry proceeds by a ser...,NaN
4,A Fire Upon the Deep,Vernor Vinge,NaN,The novel posits that space around the Milky ...,NaN
...,...,...,...,...,...
175545,Coaching Outside the Box,Paul Mairs,2012-01-01,This book demonstrates what coaches should and...,171
175546,The women's wheel of life,"Elizabeth Davis,Carol Leonard",1997-04,For women who have found identity in nonconven...,240
175547,"Gilded New York: Design, Fashion, and Society","Jeannine Falino,Phyllis Magidson,Nina Gray,Don...",2013-11-05,The Gilded Years of the late nineteenth centur...,240
175548,Student Power in World Evangelism,David M. Howard Jr.,1970-01-01,"When students decide to act, things happen. Th...",129


In [96]:
tqdm.pandas()

In [97]:
book_dataset_dict = {}
for i, row in tqdm(book_dataset.iterrows()):
    book_dataset_dict[(row.title, row.publication_date)] = row.author

95355it [00:06, 15135.39it/s]


In [98]:
book_dataset_dict[("One Nation Under God: Religious Symbols, Quotes, and Images in Our Nation's Capital", '2001-04-01')]

' Eugene F. Hemrick'

In [100]:
def get_new_author(title, publication_date):
    return book_dataset_dict.get((title, publication_date), "NO")

books_prepared.loc[
    (books_prepared.title.isin(book_dataset.title))&
     (books_prepared.publication_date.isin(book_dataset.publication_date)),
    'new_author'
] = books_prepared.loc[
    (books_prepared.title.isin(book_dataset.title))&
     (books_prepared.publication_date.isin(book_dataset.publication_date)),
].progress_apply(lambda row: get_new_author(row.title, row.publication_date), axis=1)

100%|██████████| 68161/68161 [00:01<00:00, 44219.19it/s]


In [104]:
books_prepared.shape

(175550, 6)

In [106]:
books_prepared['author'] = np.where(
    books_prepared['new_author'].notna(),  # условие: если не NaN
    books_prepared['new_author'],          # тогда берем значение из нового столбца
    books_prepared['author']           # иначе оставляем старое значение
)

In [112]:
books_prepared = books_prepared.loc[books_prepared.author!='NO']

In [115]:
books_prepared[['title', 'author', 'publication_date', 'description', 'page_count']].to_excel(datapath + 'book_data_prepared.xlsx')

In [116]:
books_prepared.shape

(174467, 6)